In [ ]:
# 1. Clean out any corrupted cached versions first
!pip uninstall xformers bitsandbytes unsloth -y

# 2. Install the new, stabilized direct package
!pip install unsloth

# 3. Pull the absolute latest bleeding-edge updates directly from source
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 4. Install supporting structural frameworks
!pip install datasets trl transformers accelerate

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: unsloth 2026.5.2
Uninstalling unsloth-2026.5.2:
  Successfully uninstalled unsloth-2026.5.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 1.5 MB/s eta 0:00:00
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 9.4 MB/s eta 0:00:00
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 44.0 MB/s eta 0:00:00
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-sjd4ecu1/unsloth_43e4f8df2eea4fceaec729b5ce25dc13
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-sjd4ecu1/unsloth_43e4f8df2eea4fceaec729b5ce25dc13
  Resolved https://github.com/unslothai/unsloth.git to commit e77

In [ ]:
import torch
import os
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Base Configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Verification of dataset file
data_path = "/content/metrostack_train.jsonl"
if not os.path.exists(data_path):
    raise FileNotFoundError(f"DATASET MISSING: Please upload '{data_path}' to the Colab file sidebar (folder icon on the left) before running this cell.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Setup LoRA Target Parameters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

# 3. Format & Apply ChatML Templates
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files=data_path, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 4. Define the Fine-Tuning Hyperparameters
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 5. Cook the Weights
print("--- STARTING METROBOT TRAINING LOOP ---")
trainer_stats = trainer.train()

# 6. Direct GGUF Extraction
print("--- PACKAGING COMPLETED MODEL TO GGUF ---")
# Ensuring the output format is explicitly defined
model.save_pretrained_gguf("metrobot_gguf", tokenizer, quantization_method = "q4_k_m")
print("--- PIPELINE SUCCESSFUL. YOU CAN NOW DOWNLOAD THE GGUF FILE FROM THE SIDEBAR ---")

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/35 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:04<00:00, 64.57s/it]


Unsloth: Merge process complete. Saved to `/content/metrobot_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


ERROR:unsloth_zoo.llama_cpp:Unsloth: Error during loading or introspecting the original script: Failed to execute module convert_hf_to_gguf_original_gguf_grj8n_t9 from /root/.unsloth/llama.cpp/original_gguf_grj8n_t9.py
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 894, in _load_module_from_path
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/root/.unsloth/llama.cpp/original_gguf_grj8n_t9.py", line 18, in <module>
    from conversion import (
ModuleNotFoundError: No module named 'conversion'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 957, in _download_convert_hf_to_gguf_cached
    module = _load_module_from_path(temp_original_file_

RuntimeError: Unsloth: GGUF conversion failed: Failed during loading/introspection of original script: Failed to execute module convert_hf_to_gguf_original_gguf_grj8n_t9 from /root/.unsloth/llama.cpp/original_gguf_grj8n_t9.py

In [ ]:
# 1. Install compatible dependencies for GGUF conversion
!pip install "protobuf<5.0.0" gguf

# 2. Re-attempt the conversion with robust settings
print("--- RE-ATTEMPTING GGUF CONVERSION ---")
try:
    import os
    os.environ["UNSLOTH_GGUF_USE_FAST"] = "1"

    model.save_pretrained_gguf(
        "metrobot_gguf",
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print("--- CONVERSION FINISHED ---")
    print("The .gguf file should now appear in the 'metrobot_gguf' folder in the left sidebar.")
except Exception as e:
    print(f"Conversion failed again: {e}")
    print("Checking for any generated files:")
    !ls -R metrobot_gguf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 23.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.


--- RE-ATTEMPTING GGUF CONVERSION ---
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 6765.01it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:28<00:00, 88.62s/it]


Unsloth: Merge process complete. Saved to `/content/metrobot_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


ERROR:unsloth_zoo.llama_cpp:Unsloth: Error during loading or introspecting the original script: Failed to execute module convert_hf_to_gguf_original_gguf_bqppavcx from /root/.unsloth/llama.cpp/original_gguf_bqppavcx.py
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 894, in _load_module_from_path
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/root/.unsloth/llama.cpp/original_gguf_bqppavcx.py", line 18, in <module>
    from conversion import (
ModuleNotFoundError: No module named 'conversion'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 957, in _download_convert_hf_to_gguf_cached
    module = _load_module_from_path(temp_original_file_

Conversion failed again: Unsloth: GGUF conversion failed: Failed during loading/introspection of original script: Failed to execute module convert_hf_to_gguf_original_gguf_bqppavcx from /root/.unsloth/llama.cpp/original_gguf_bqppavcx.py
Checking for any generated files:
metrobot_gguf:
chat_template.jinja  model.safetensors	    tokenizer.json
config.json	     tokenizer_config.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from unsloth import FastLanguageModel
import torch

# 1. Load the base model and tokenizer from the clean hub source
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 2. Load the fine-tuned adapters from the training output directory
# This ensures we use the correct weights without the corrupted merged config
model = FastLanguageModel.for_inference(model)
model.load_adapter("outputs/checkpoint-60") # Using the final checkpoint

# 3. Configure padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4. Define the test prompt
messages = [
    {"role": "user", "content": "How do I query the MetroStack database for recent spatial updates?"},
]

# 5. Generate response
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 0.7,
    top_p = 0.9,
    repetition_penalty = 1.2
)

response = tokenizer.batch_decode(outputs, skip_special_tokens = True)

print("--- TEST INFERENCE RESPONSE ---")
print(response[0])

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


--- TEST INFERENCE RESPONSE ---
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
How do I query the MetroStack database for recent spatial updates?
assistant
To access recent spatial updates in your local MetroStack environment, you can use the following SQL command:

```sql
SELECT 
    id,
    updated_at,
    CASE WHEN deleted = FALSE THEN 'Active' ELSE 'Deleted' END AS status
FROM 
    space
ORDER BY 
    updated_at DESC;
```

This query retrieves unique identifiers (`id`), timestamps of last update (`updated_at`), and whether each feature is currently active or marked as deleted ('Active' or 'Deleted'). The results are ordered from most recently updated to least. This allows you to see what features have been modified or removed within the specified time


### Steps to Import into Ollama

1. **Download the GGUF file**: Locate the `.gguf` file in the `/content/metrobot_gguf` folder and download it to your local machine.
2. **Create a Modelfile**: Create a plain text file named `Modelfile` (no extension) in the same folder where you saved the GGUF.
3. **Add the configuration**: Use the content generated below.
4. **Run the Import Command**: Open your terminal and run:
   ```bash
   ollama create metrobot -f Modelfile
   ```

In [17]:
# Helper to print the exact Modelfile content based on your GGUF name
import os

gguf_folder = "/content/metrobot_gguf"
files = [f for f in os.listdir(gguf_folder) if f.endswith(".gguf")]
gguf_filename = files[0] if files else "unsloth.Q4_K_M.gguf"

modelfile_content = f"""FROM ./{gguf_filename}

# Sets the temperature to 0.7 [Higher is more creative, lower is more coherent]
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop <|im_start|>
PARAMETER stop <|im_end|>

# System prompt matches your training
SYSTEM \"\"\"You are Qwen, created by Alibaba Cloud. You are a helpful assistant specialized in MetroStack.\"\"\"
"""

print("--- COPY THIS CONTENT INTO YOUR LOCAL Modelfile ---")
print(modelfile_content)

--- COPY THIS CONTENT INTO YOUR LOCAL Modelfile ---
FROM ./unsloth.Q4_K_M.gguf

# Sets the temperature to 0.7 [Higher is more creative, lower is more coherent]
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop <|im_start|>
PARAMETER stop <|im_end|>

# System prompt matches your training
SYSTEM """You are Qwen, created by Alibaba Cloud. You are a helpful assistant specialized in MetroStack."""



In [18]:
model.save_pretrained_gguf("/content/metrobot_gguf", tokenizer, quantization_method = "q4_k_m")

Unsloth: Model is not a PEFT model. Saving directly without LoRA merge...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/metrobot_gguf/tokenizer_config.json.


Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


ERROR:unsloth_zoo.llama_cpp:Unsloth: Error during loading or introspecting the original script: Failed to execute module convert_hf_to_gguf_original_gguf_d69qn7dg from /root/.unsloth/llama.cpp/original_gguf_d69qn7dg.py
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 894, in _load_module_from_path
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/root/.unsloth/llama.cpp/original_gguf_d69qn7dg.py", line 18, in <module>
    from conversion import (
ModuleNotFoundError: No module named 'conversion'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 957, in _download_convert_hf_to_gguf_cached
    module = _load_module_from_path(temp_original_file_

RuntimeError: Unsloth: GGUF conversion failed: Failed during loading/introspection of original script: Failed to execute module convert_hf_to_gguf_original_gguf_d69qn7dg from /root/.unsloth/llama.cpp/original_gguf_d69qn7dg.py

In [19]:
%%bash
# 1. Move to the root directory and get the conversion tools
cd /content
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
pip install -r requirements.txt

# 2. Build the quantize tool (this takes a minute or two)
make quantize

# 3. Create your output directory
mkdir -p /content/metrobot_gguf

# 4. Convert the raw files in /content into an unquantized F16 GGUF
python convert_hf_to_gguf.py /content --outfile /content/metrobot-f16.gguf --outtype f16

# 5. Quantize the model to Q4_K_M and output it to your target folder
./llama-quantize /content/metrobot-f16.gguf /content/metrobot_gguf/metrobot-Q4_K_M.gguf q4_k_m

# 6. Clean up the massive F16 file to free up Colab disk space
rm /content/metrobot-f16.gguf

echo "Conversion complete! Refresh your file panel to see metrobot-Q4_K_M.gguf"

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 69.7 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.6/1

Cloning into 'llama.cpp'...
Updating files: 100% (2849/2849), done.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.5.2 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.5.1 which is incompatible.
unsloth-zoo 2026.5.1 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.5.1 which is incompatible.
xformers 0.0.35 requires torch>=2.10, but you have torch 2.6.0+cpu which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
opencv-python-headless 4.13.0.9

### Clear `unsloth_compiled_cache` and Retry GGUF Conversion

In [21]:
import shutil
import os
import sys

# Clear the unsloth_compiled_cache folder
cache_path = "/content/unsloth_compiled_cache"
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print(f"Cleared cache: {cache_path}")
else:
    print(f"Cache directory not found: {cache_path}")

# Add unsloth's internal llama.cpp directory to sys.path
# This is where unsloth's copied conversion scripts are likely located
# and where they expect to find the 'conversion' module.
unsloth_llama_cpp_path = "/root/.unsloth/llama.cpp"
if os.path.exists(unsloth_llama_cpp_path) and unsloth_llama_cpp_path not in sys.path:
    sys.path.insert(0, unsloth_llama_cpp_path)
    print(f"Added {unsloth_llama_cpp_path} to sys.path")
else:
    print(f"Path {unsloth_llama_cpp_path} not found or already in sys.path.")

# Retry the GGUF conversion
print("--- RETRYING GGUF CONVERSION AFTER CLEARING CACHE AND ADJUSTING PATH ---")
try:
    # Ensure environment variable is set for fast conversion if needed
    os.environ["UNSLOTH_GGUF_USE_FAST"] = "1"

    model.save_pretrained_gguf(
        "metrobot_gguf",
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print("--- CONVERSION FINISHED SUCCESSFULLY ---")
    print("The .gguf file should now appear in the 'metrobot_gguf' folder in the left sidebar.")
except Exception as e:
    print(f"Conversion failed again after cache clear and path adjustment: {e}")
    print("Checking for any generated files:")
    !ls -R metrobot_gguf

Cache directory not found: /content/unsloth_compiled_cache
Added /root/.unsloth/llama.cpp to sys.path
--- RETRYING GGUF CONVERSION AFTER CLEARING CACHE AND ADJUSTING PATH ---
Unsloth: Model is not a PEFT model. Saving directly without LoRA merge...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['metrobot_gguf_gguf/qwen2.5-coder-1.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['metrobot_gguf_gguf/qwen2.5-coder-1.5b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model metrobot_gguf_gguf/qwen2.5-coder-1.5b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to metrobot_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f metrobot_gguf_gguf/Modelfile
--- CONVERSION FINISHED SUCCESSFULLY ---
The .gguf file should now appear in the 'metrobot_gguf' folder in the left sidebar.
